# Stream C - Stance Scoring (v3, NLI-based)

Computes stance features by treating evidence as the **premise** and the caption as the **hypothesis** in a Natural Language Inference (NLI) model.

**Key changes vs. previous version:**
1. **Replaces cosine similarity with NLI** (entailment / neutral / contradiction logits).
2. NLI is the principled formulation for evidence-based fact verification (used in FEVER, ANLI, MultiNLI literature).
3. Uses `microsoft/deberta-v3-base-mnli` by default (small enough for 8 GB VRAM, strong on NLI).
4. SBERT cosine kept as a fallback feature for robustness/comparison.
5. Outputs the same column names (`support_score`, `contradiction_score`, `neutral_score`) as before so `stream_c_results` notebook needs no changes.

**Runtime estimate (RTX 4060 Ti, 8 GB VRAM):**
- Setup + load: ~2 min
- NLI scoring 97k rows @ batch 32, mixed precision: ~25–40 min
- SBERT cosine 97k rows @ batch 256: ~5 min
- Output writing: ~1 min
- **Total: ~35–50 min full run**

**DEBUG mode** (set `DEBUG=True`) runs on 256 rows in ~1 min for sanity checking.


In [11]:
from pathlib import Path
import sys, os, json, pickle, re, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import STREAM_C_CACHE, STREAM_A_CACHE, STREAM_B_CACHE, ALL_MANIFEST, FIGURES, TABLES

STREAM_C_CACHE.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("STREAM_C_CACHE:", STREAM_C_CACHE)


ROOT: E:\eve-rag
STREAM_C_CACHE: E:\eve-rag\cache\stream_c


In [12]:
# === DEBUG MODE ===
# Set DEBUG=True for a fast smoke test on 256 rows. Always run DEBUG=True first to
# verify the notebook runs end-to-end, then set DEBUG=False for the full run.
DEBUG = False
DEBUG_SAMPLES = 256

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

# Batch sizes tuned for 8 GB VRAM. Lower if OOM.
NLI_BATCH_SIZE = 32 if DEVICE == "cuda" else 8
SBERT_BATCH_SIZE = 256 if DEVICE == "cuda" else 64
NLI_MAX_LEN = 512
print(f"DEBUG={DEBUG}, NLI_BATCH={NLI_BATCH_SIZE}, SBERT_BATCH={SBERT_BATCH_SIZE}")


Torch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM GB: 8.0
DEBUG=False, NLI_BATCH=32, SBERT_BATCH=256


In [13]:
# === Load evidence ===
EVIDENCE_DIR = STREAM_C_CACHE / "evidence"
STANCE_DIR = STREAM_C_CACHE / "stance_scores"
STANCE_DIR.mkdir(parents=True, exist_ok=True)
evidence_path = EVIDENCE_DIR / "raw_evidence_all.csv"
if not evidence_path.exists():
    raise FileNotFoundError(f"Missing {evidence_path}. Run evidence_retrieval_v3 first.")
evidence = pd.read_csv(evidence_path, dtype={"pair_id":"string","caption":"string","evidence_text":"string"}, low_memory=False)
evidence["caption"] = evidence["caption"].fillna("").astype(str)
evidence["evidence_text"] = evidence["evidence_text"].fillna("").astype(str)

if DEBUG:
    # Stratified sample so all dataset/split combos are represented in debug mode
    evidence = (evidence.groupby(["dataset", "split"], group_keys=False)[evidence.columns.tolist()]
                   .apply(lambda g: g.sample(min(len(g), max(1, DEBUG_SAMPLES // 6)), random_state=42))
                   .reset_index(drop=True))
    print(f"DEBUG mode: subsampled to {len(evidence)} rows")

print("Evidence rows to score:", len(evidence))
display(evidence.head())


Evidence rows to score: 97361


,pair_id,dataset,split,label,caption,evidence_text,evidence_len,caption_len
0,nc_620701_0,newsclippings,train,0,Saudi troops cheer as they ride at the back of...,"Article title: Inside Mazrak refugee camp, Yem...",250,125
1,nc_620701_1,newsclippings,train,1,Israeli soldiers ride on a tank to a position ...,Article title: Photo highlights of the day | N...,243,73
2,nc_1569820_0,newsclippings,train,0,Tiffany is north east of Aberdeen,Article title: Improvement call for CNR North ...,280,33
3,nc_1569820_1,newsclippings,train,1,Some ferry services were affected between Holy...,Article title: Warning of 'severe weather' to ...,275,71
4,nc_1490080_0,newsclippings,train,0,Rock band Paramore is another group which sell...,Article title: Whatever floats your boat - the...,282,55


In [14]:
# === Load NLI model ===
# DeBERTa-v3-base-mnli is the chosen NLI backbone:
#   - small enough for 8 GB VRAM at batch 32 with fp16
#   - strong NLI baseline (trained on MNLI; transfers well to evidence-claim verification)
#   - 3-class output: [contradiction, neutral, entailment]
# We also keep an SBERT cosine model as a complementary feature.
USE_NLI = True
nli_tokenizer = None
nli_model = None
NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-base"
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
    nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME).to(DEVICE).eval()
    if USE_AMP:
        nli_model = nli_model.half()
    print(f"Loaded NLI model: {NLI_MODEL_NAME}")
    # Determine label index mapping (varies between checkpoints).
    # DeBERTa-v3-base-mnli labels are typically:
    #   id2label = {0: 'CONTRADICTION', 1: 'NEUTRAL', 2: 'ENTAILMENT'}
    id2label = {int(k): v.upper() for k, v in nli_model.config.id2label.items()}
    print("NLI id2label:", id2label)
    # Build lookup so we always pick the right column regardless of order
    NLI_CONTRADICTION_IDX = next(i for i, l in id2label.items() if "CONTRAD" in l)
    NLI_NEUTRAL_IDX = next(i for i, l in id2label.items() if "NEUTRAL" in l)
    NLI_ENTAILMENT_IDX = next(i for i, l in id2label.items() if "ENTAIL" in l)
    print(f"contradiction={NLI_CONTRADICTION_IDX}, neutral={NLI_NEUTRAL_IDX}, entailment={NLI_ENTAILMENT_IDX}")
except Exception as e:
    USE_NLI = False
    print("NLI model unavailable; will fall back to SBERT cosine only.", repr(e))


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded NLI model: cross-encoder/nli-deberta-v3-base
NLI id2label: {0: 'CONTRADICTION', 1: 'ENTAILMENT', 2: 'NEUTRAL'}
contradiction=0, neutral=2, entailment=1


In [15]:
# === Load SBERT for cosine similarity feature ===
# We keep cosine similarity as a complementary feature alongside NLI logits.
# It captures lexical/semantic overlap that the NLI head sometimes misses.
USE_SBERT = True
sbert_model = None
SBERT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
try:
    from sentence_transformers import SentenceTransformer
    sbert_model = SentenceTransformer(SBERT_MODEL_NAME, device=DEVICE)
    print(f"Loaded SBERT model: {SBERT_MODEL_NAME}")
except Exception as e:
    USE_SBERT = False
    print("SBERT unavailable; cosine feature will be 0.", repr(e))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded SBERT model: sentence-transformers/all-MiniLM-L6-v2


In [16]:
# === NLI scoring ===
# Premise = evidence_text, hypothesis = caption.
# An out-of-context pair should produce LOW entailment and HIGH contradiction (or HIGH neutral
# when evidence is sparse). A genuine pair should produce HIGH entailment.
# We output softmax probabilities so downstream classifiers can use them directly.
import time

captions = evidence["caption"].tolist()
evtexts = evidence["evidence_text"].tolist()
N = len(captions)

nli_entail = np.zeros(N, dtype=np.float32)
nli_neutral = np.zeros(N, dtype=np.float32)
nli_contradict = np.zeros(N, dtype=np.float32)

if USE_NLI:
    t0 = time.time()
    with torch.no_grad():
        for start in range(0, N, NLI_BATCH_SIZE):
            end = min(start + NLI_BATCH_SIZE, N)
            premises = evtexts[start:end]
            hypotheses = captions[start:end]
            # Truncate aggressively on the premise side because evidence can be long.
            enc = nli_tokenizer(
                premises, hypotheses,
                padding=True, truncation="only_first",
                max_length=NLI_MAX_LEN, return_tensors="pt"
            ).to(DEVICE)
            logits = nli_model(**enc).logits.float()
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            nli_contradict[start:end] = probs[:, NLI_CONTRADICTION_IDX]
            nli_neutral[start:end] = probs[:, NLI_NEUTRAL_IDX]
            nli_entail[start:end] = probs[:, NLI_ENTAILMENT_IDX]
            if start % (NLI_BATCH_SIZE * 50) == 0:
                elapsed = time.time() - t0
                rate = (end) / max(elapsed, 0.01)
                eta = (N - end) / max(rate, 0.01)
                print(f"  NLI {end}/{N}  ({rate:.1f} rows/s, ETA {eta/60:.1f} min)")
    print(f"NLI scoring done in {(time.time()-t0)/60:.2f} min")
    # Free VRAM before SBERT
    del nli_model
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
else:
    # If NLI unavailable, leave arrays as zeros and rely on SBERT cosine for support_score
    print("Skipping NLI (model unavailable)")


  NLI 32/97361  (638.6 rows/s, ETA 2.5 min)
  NLI 1632/97361  (659.2 rows/s, ETA 2.4 min)
  NLI 3232/97361  (695.9 rows/s, ETA 2.3 min)
  NLI 4832/97361  (708.7 rows/s, ETA 2.2 min)
  NLI 6432/97361  (716.9 rows/s, ETA 2.1 min)
  NLI 8032/97361  (720.0 rows/s, ETA 2.1 min)
  NLI 9632/97361  (724.0 rows/s, ETA 2.0 min)
  NLI 11232/97361  (725.6 rows/s, ETA 2.0 min)
  NLI 12832/97361  (727.2 rows/s, ETA 1.9 min)
  NLI 14432/97361  (728.3 rows/s, ETA 1.9 min)
  NLI 16032/97361  (729.6 rows/s, ETA 1.9 min)
  NLI 17632/97361  (730.4 rows/s, ETA 1.8 min)
  NLI 19232/97361  (730.4 rows/s, ETA 1.8 min)
  NLI 20832/97361  (730.7 rows/s, ETA 1.7 min)
  NLI 22432/97361  (731.1 rows/s, ETA 1.7 min)
  NLI 24032/97361  (730.9 rows/s, ETA 1.7 min)
  NLI 25632/97361  (730.9 rows/s, ETA 1.6 min)
  NLI 27232/97361  (730.9 rows/s, ETA 1.6 min)
  NLI 28832/97361  (730.5 rows/s, ETA 1.6 min)
  NLI 30432/97361  (729.0 rows/s, ETA 1.5 min)
  NLI 32032/97361  (728.0 rows/s, ETA 1.5 min)
  NLI 33632/97361  (72

In [17]:
# === SBERT cosine feature ===
# Symmetric semantic similarity between caption and evidence.
# This complements the directional NLI signal.
sbert_sim = np.zeros(N, dtype=np.float32)
if USE_SBERT:
    t0 = time.time()
    cap_emb = sbert_model.encode(captions, batch_size=SBERT_BATCH_SIZE, show_progress_bar=True,
                                  convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    ev_emb = sbert_model.encode(evtexts, batch_size=SBERT_BATCH_SIZE, show_progress_bar=True,
                                 convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    sbert_sim = np.sum(cap_emb * ev_emb, axis=1).astype(np.float32)
    print(f"SBERT cosine done in {(time.time()-t0)/60:.2f} min")
    del sbert_model, cap_emb, ev_emb
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
else:
    print("Skipping SBERT cosine (model unavailable)")


Batches:   0%|          | 0/381 [00:00<?, ?it/s]

Batches:   0%|          | 0/381 [00:00<?, ?it/s]

SBERT cosine done in 0.61 min


In [18]:
# === Build output dataframe (robust to missing dataset/split) ===
# If evidence somehow lost dataset/split (e.g., a stale CSV write), re-merge from manifest.
required = {"pair_id", "dataset", "split", "label", "caption", "evidence_text"}
missing_cols = required - set(evidence.columns)
if missing_cols:
    print(f"WARNING: evidence is missing {missing_cols}; re-merging from manifest")
    manifest = pd.read_csv(ALL_MANIFEST,
        dtype={"pair_id":"string","dataset":"string","split":"string","caption":"string"},
        low_memory=False)[["pair_id","dataset","split","label","caption"]]
    # Keep only the columns we have from evidence to avoid duplicate suffixes
    keep_from_ev = [c for c in evidence.columns if c not in {"dataset","split","label","caption"}]
    evidence = manifest.merge(evidence[keep_from_ev], on="pair_id", how="inner")
    print(f"After re-merge: {len(evidence)} rows, columns: {list(evidence.columns)}")

df = evidence[["pair_id", "dataset", "split", "label", "caption", "evidence_text"]].copy()

if USE_NLI:
    df["support_score"] = nli_entail
    df["contradiction_score"] = nli_contradict
    df["neutral_score"] = nli_neutral
else:
    df["support_score"] = sbert_sim
    df["contradiction_score"] = 1.0 - sbert_sim
    df["neutral_score"] = np.zeros(N, dtype=np.float32)

df["stance_similarity"] = sbert_sim if USE_SBERT else nli_entail
df["caption_len"] = df["caption"].fillna("").astype(str).str.len()
df["evidence_len"] = df["evidence_text"].fillna("").astype(str).str.len()
df["len_ratio"] = df["caption_len"] / (df["evidence_len"] + 1)

if not DEBUG:
    df.to_csv(STANCE_DIR / "stream_c_stance_scores_all.csv", index=False)
    pickle.dump(df, open(STANCE_DIR / "stream_c_stance_scores_all.pkl", "wb"), protocol=4)
    for (dataset, split), sub in df.groupby(["dataset", "split"]):
        sub.to_csv(STANCE_DIR / f"stream_c_stance_scores_{dataset}_{split}.csv", index=False)
    print("Saved full-run stance scores")
else:
    print("DEBUG mode: skipped saving (set DEBUG=False and re-run for full output)")

display(df.head())

Saved full-run stance scores


,pair_id,dataset,split,label,caption,evidence_text,support_score,contradiction_score,neutral_score,stance_similarity,caption_len,evidence_len,len_ratio
0,nc_620701_0,newsclippings,train,0,Saudi troops cheer as they ride at the back of...,"Article title: Inside Mazrak refugee camp, Yem...",0.001510,0.990787,0.007703,0.399399,125,250,0.498008
1,nc_620701_1,newsclippings,train,1,Israeli soldiers ride on a tank to a position ...,Article title: Photo highlights of the day | N...,0.006170,0.986299,0.007530,0.079025,73,243,0.299180
2,nc_1569820_0,newsclippings,train,0,Tiffany is north east of Aberdeen,Article title: Improvement call for CNR North ...,0.000151,0.000170,0.999679,0.335277,33,280,0.117438
3,nc_1569820_1,newsclippings,train,1,Some ferry services were affected between Holy...,Article title: Warning of 'severe weather' to ...,0.000350,0.000523,0.999127,0.219046,71,275,0.257246
4,nc_1490080_0,newsclippings,train,0,Rock band Paramore is another group which sell...,Article title: Whatever floats your boat - the...,0.000048,0.001151,0.998800,0.377815,55,282,0.194346


In [19]:
# === Summary ===
summary = df.groupby(["dataset", "split"]).agg(
    rows=("pair_id", "count"),
    support_mean=("support_score", "mean"),
    contradict_mean=("contradiction_score", "mean"),
    neutral_mean=("neutral_score", "mean"),
    sim_mean=("stance_similarity", "mean")
).reset_index()
if not DEBUG:
    summary.to_csv(TABLES / "stream_c_stance_summary.csv", index=False)
display(summary)

# Quick label-conditional sanity check: for each dataset, check whether support_score
# differs between match (label=0) and OOC (label=1). If decircularization worked, we
# expect a non-zero gap (genuine pairs should entail evidence more than OOC pairs).
gap = df.groupby(["dataset", "split", "label"])["support_score"].mean().unstack("label")
gap.columns = [f"label_{c}" for c in gap.columns]
gap["gap_match_minus_ooc"] = gap.get("label_0", 0) - gap.get("label_1", 0)
display(gap.round(4))
print("If gap_match_minus_ooc is meaningfully positive, NLI is providing real signal.")
print("Stance scoring v3 complete.")


,dataset,split,rows,support_mean,contradict_mean,neutral_mean,sim_mean
0,mmfakebench,test,10000,0.137893,0.630257,0.231850,0.046476
1,mmfakebench,val,1000,0.157474,0.612510,0.230016,0.047998
2,newsclippings,test,7264,0.034681,0.343750,0.621569,0.365072
3,newsclippings,train,71072,0.033704,0.359807,0.606488,0.369728
4,newsclippings,val,7024,0.038844,0.351811,0.609345,0.368992
5,verite,test,1001,0.001029,0.425442,0.573529,0.138622


label_0  label_1  gap_match_minus_ooc
dataset       split                                       
mmfakebench   test    0.1163   0.1471              -0.0308
              val     0.1720   0.1513               0.0207
newsclippings test    0.0345   0.0349              -0.0004
              train   0.0339   0.0335               0.0004
              val     0.0421   0.0356               0.0065
verite        test    0.0010   0.0010              -0.0001

If gap_match_minus_ooc is meaningfully positive, NLI is providing real signal.
Stance scoring v3 complete.


In [20]:
print("Columns:", list(evidence.columns))
print("Shape:", evidence.shape)
print(evidence.head(2))

Columns: ['pair_id', 'dataset', 'split', 'label', 'caption', 'evidence_text', 'evidence_len', 'caption_len']
Shape: (97361, 8)
       pair_id        dataset  split  label  \
0  nc_620701_0  newsclippings  train      0   
1  nc_620701_1  newsclippings  train      1   

                                             caption  \
0  Saudi troops cheer as they ride at the back of...   
1  Israeli soldiers ride on a tank to a position ...   

                                       evidence_text  evidence_len  \
0  Article title: Inside Mazrak refugee camp, Yem...           250   
1  Article title: Photo highlights of the day | N...           243   

   caption_len  
0          125  
1           73  
